# H5 to CSVs

Convert the `1001Prostate` HDF5 modality files into patient-level CSVs aligned with the style used for `MIMM` and `mmCRC`.

## Scope

This notebook builds one CSV per modality:

- `radiology_1001prostate.csv` from `CT_BL`
- `radioreports_1001prostate.csv` from `CT_report_BL`
- `blood_1001prostate.csv` from `lab_test`

If any explicit OS/endpoints source is found, it also writes `endpoints_1001prostate.csv`.

Implementation details:

- `CT_BL` and `lab_test` already have at most one valid file per patient.
- `CT_report_BL` may contain multiple files per patient; the notebook keeps the valid file with the smallest `|time|`, i.e. the report closest to diagnosis.
- Empty or invalid HDF5 files are skipped and logged.
- The embedding dataset is expected at `/feats`.
- `lab_test` uses `/feat_types` to name columns directly instead of generic `embedding_*` names.

In [ ]:
from __future__ import annotations

import math
import os
import re
import subprocess
from collections import Counter, defaultdict
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
from IPython.display import display

DATA_ROOT = Path('/Users/marcalbesa/Desktop/TFM/data/1001Prostate')
OUTPUT_DIR = DATA_ROOT

MODALITY_TO_FILENAME = {
    'CT_BL': 'radiology_1001prostate.csv',
    'CT_report_BL': 'radioreports_1001prostate.csv',
    'lab_test': 'blood_1001prostate.csv',
}

ENDPOINTS_FILENAME = 'endpoints_1001prostate.csv'

assert DATA_ROOT.exists(), DATA_ROOT


In [ ]:
TOKEN_RE = re.compile(r'nan|[-+]?(?:\d+\.\d*|\d*\.\d+|\d+)(?:[eE][-+]?\d+)?', re.IGNORECASE)


def run_cmd(args: list[str]) -> str:
    return subprocess.check_output(args, text=True, stderr=subprocess.DEVNULL)


def h5_dataset_names(path: Path) -> list[str]:
    out = run_cmd(['h5ls', '-r', str(path)])
    names: list[str] = []
    for line in out.splitlines():
        line = line.strip()
        if line.startswith('/'):
            names.append(line.split()[0])
    return names


def parse_numeric_dataset(path: Path, dataset: str) -> np.ndarray:
    dump = run_cmd(['h5dump', '-d', dataset, str(path)])
    data_part = dump.split('DATA {', 1)[1].rsplit('}', 1)[0]
    data_part = re.sub(r'\([^)]*\):', ' ', data_part)
    tokens = TOKEN_RE.findall(data_part)
    values = [float('nan') if token.lower() == 'nan' else float(token) for token in tokens]
    return np.asarray(values, dtype=float)


def parse_scalar_text_dataset(path: Path, dataset: str) -> str:
    dump = run_cmd(['h5dump', '-w', '0', '-d', dataset, str(path)])
    start_marker = '(0): "'
    start = dump.find(start_marker)
    end = dump.rfind('"')
    if start == -1 or end == -1 or end <= start + len(start_marker):
        raise ValueError(f'Could not parse text dataset {dataset} from {path}')
    return dump[start + len(start_marker):end]


def parse_scalar_float_dataset(path: Path, dataset: str) -> float:
    values = parse_numeric_dataset(path, dataset)
    if values.size != 1:
        raise ValueError(f'Expected scalar dataset {dataset} in {path}, got {values.size} values')
    return float(values[0])


def patient_from_path(path: Path) -> str:
    return path.name.split('_')[0]


def study_date_from_path(path: Path) -> str:
    return path.stem.rsplit('_', 1)[-1]


def sanitize_column_name(name: str) -> str:
    name = name.strip().lower()
    name = re.sub(r'[^a-z0-9]+', '_', name)
    name = re.sub(r'_+', '_', name).strip('_')
    return name


In [ ]:
def summarize_modality_tree(modality_dir: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(modality_dir.glob('*.h5')):
        dataset_names = h5_dataset_names(path)
        rows.append({
            'patient': patient_from_path(path),
            'source_file': path.name,
            'dataset_names': ', '.join(dataset_names),
        })
    return pd.DataFrame(rows)

scan_frames = {}
for modality in MODALITY_TO_FILENAME:
    scan_frames[modality] = summarize_modality_tree(DATA_ROOT / modality)
    print(modality)
    display(scan_frames[modality].head())
    print('files:', len(scan_frames[modality]), 'patients:', scan_frames[modality]['patient'].nunique())
    print()


In [ ]:
def build_radiology_df() -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    invalid_rows = []
    for path in sorted((DATA_ROOT / 'CT_BL').glob('*.h5')):
        dataset_names = set(h5_dataset_names(path))
        if not {'/feats', '/time'}.issubset(dataset_names):
            invalid_rows.append({'modality': 'CT_BL', 'source_file': path.name, 'reason': 'missing_required_dataset'})
            continue
        feats = parse_numeric_dataset(path, '/feats').reshape(-1)
        time_value = parse_scalar_float_dataset(path, '/time')
        row = {
            'patient': patient_from_path(path),
            'study_date': study_date_from_path(path),
            'time_to_diagnosis': time_value,
            'source_file': path.name,
        }
        row.update({f'embedding_{idx}': float(value) for idx, value in enumerate(feats)})
        rows.append(row)
    return pd.DataFrame(rows), pd.DataFrame(invalid_rows)


def build_radioreports_df() -> tuple[pd.DataFrame, pd.DataFrame]:
    grouped: dict[str, list[dict]] = defaultdict(list)
    invalid_rows = []
    for path in sorted((DATA_ROOT / 'CT_report_BL').glob('*.h5')):
        dataset_names = set(h5_dataset_names(path))
        if not {'/feats', '/text', '/time'}.issubset(dataset_names):
            invalid_rows.append({'modality': 'CT_report_BL', 'source_file': path.name, 'reason': 'missing_required_dataset'})
            continue
        feats = parse_numeric_dataset(path, '/feats').reshape(-1)
        time_value = parse_scalar_float_dataset(path, '/time')
        text_value = parse_scalar_text_dataset(path, '/text')
        grouped[patient_from_path(path)].append({
            'patient': patient_from_path(path),
            'study_date': study_date_from_path(path),
            'time_to_diagnosis': time_value,
            'source_file': path.name,
            'report_text': text_value,
            'feats': feats,
        })

    rows = []
    for patient, candidates in sorted(grouped.items()):
        selected = sorted(
            candidates,
            key=lambda row: (abs(row['time_to_diagnosis']), row['time_to_diagnosis'], row['source_file'])
        )[0]
        row = {
            'patient': selected['patient'],
            'study_date': selected['study_date'],
            'time_to_diagnosis': selected['time_to_diagnosis'],
            'source_file': selected['source_file'],
            'report_text': selected['report_text'],
        }
        row.update({f'embedding_{idx}': float(value) for idx, value in enumerate(selected['feats'])})
        rows.append(row)

    return pd.DataFrame(rows), pd.DataFrame(invalid_rows)


def build_blood_df() -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    invalid_rows = []
    expected_feature_names: list[str] | None = None
    for path in sorted((DATA_ROOT / 'lab_test').glob('*.h5')):
        dataset_names = set(h5_dataset_names(path))
        if not {'/feats', '/feat_types', '/time'}.issubset(dataset_names):
            invalid_rows.append({'modality': 'lab_test', 'source_file': path.name, 'reason': 'missing_required_dataset'})
            continue
        feats = parse_numeric_dataset(path, '/feats').reshape(-1)
        feat_types = [
            item for item in re.findall(r'"([^"]+)"', run_cmd(['h5dump', '-d', '/feat_types', str(path)]))
            if item not in {str(path), '/feat_types'}
        ]
        feature_names = [sanitize_column_name(name) for name in feat_types]
        if expected_feature_names is None:
            expected_feature_names = feature_names
        elif feature_names != expected_feature_names:
            invalid_rows.append({'modality': 'lab_test', 'source_file': path.name, 'reason': 'feat_types_mismatch'})
            continue
        if len(feature_names) != len(feats):
            invalid_rows.append({'modality': 'lab_test', 'source_file': path.name, 'reason': 'feat_length_mismatch'})
            continue
        row = {
            'patient': patient_from_path(path),
            'study_date': study_date_from_path(path),
            'time_to_diagnosis': parse_scalar_float_dataset(path, '/time'),
            'source_file': path.name,
        }
        row.update({feature_name: float(value) for feature_name, value in zip(feature_names, feats)})
        rows.append(row)
    return pd.DataFrame(rows), pd.DataFrame(invalid_rows)


In [ ]:
radiology_df, radiology_invalid_df = build_radiology_df()
radioreports_df, radioreports_invalid_df = build_radioreports_df()
blood_df, blood_invalid_df = build_blood_df()

print('radiology rows:', radiology_df.shape)
print('radioreports rows:', radioreports_df.shape)
print('blood rows:', blood_df.shape)
print()
print('invalid radiology files:', len(radiology_invalid_df))
print('invalid radioreport files:', len(radioreports_invalid_df))
print('invalid blood files:', len(blood_invalid_df))


In [ ]:
display(radiology_df.head(2))
display(radioreports_df[['patient', 'study_date', 'time_to_diagnosis', 'source_file']].head(5))
display(blood_df.head(5))

if not radioreports_invalid_df.empty:
    display(radioreports_invalid_df)


In [ ]:
def search_endpoint_candidates() -> list[Path]:
    patterns = ('*os*', '*surv*', '*endpoint*', '*label*', '*.csv', '*.xlsx', '*.xls', '*.tsv', '*.json', '*.txt')
    excluded_names = set(MODALITY_TO_FILENAME.values()) | {ENDPOINTS_FILENAME}
    candidates = []
    for pattern in patterns:
        candidates.extend(DATA_ROOT.rglob(pattern))
    return sorted({
        path for path in candidates
        if path.is_file() and path.suffix.lower() != '.h5' and path.name not in excluded_names
    })

endpoint_candidates = search_endpoint_candidates()
endpoint_candidates


## Save CSVs

This writes the modality CSVs directly into `DATA_ROOT`. No endpoints CSV is written unless an explicit OS/endpoints source is found.

In [ ]:
radiology_out = OUTPUT_DIR / MODALITY_TO_FILENAME['CT_BL']
radioreports_out = OUTPUT_DIR / MODALITY_TO_FILENAME['CT_report_BL']
blood_out = OUTPUT_DIR / MODALITY_TO_FILENAME['lab_test']

radiology_df.to_csv(radiology_out, index=False)
radioreports_df.to_csv(radioreports_out, index=False)
blood_df.to_csv(blood_out, index=False)

print('saved', radiology_out)
print('saved', radioreports_out)
print('saved', blood_out)

if endpoint_candidates:
    print('Endpoint candidates found. Build an endpoints CSV manually from one of these sources:')
    for path in endpoint_candidates:
        print('-', path)
else:
    print('No explicit OS/endpoints source found under DATA_ROOT; endpoints CSV not created.')
